# Using Features to Predict the Risk of Heart Disease

The data for this exercise is available here (https://archive.ics.uci.edu/ml/datasets/heart+Disease) (in CSV format).

The dataset contains information from 303 patients who may or may not have heart disease. The features for each patient are:

Column | Description | Feature Type
------------ | -------------------- | ----------------------
Age | Age in years | Numerical
Sex | (1 = male; 0 = female) | Categorical
CP | Chest pain type (0, 1, 2, 3, 4) | Categorical
Trestbpd | Resting blood pressure (in mm Hg on admission) | Numerical
Chol | Serum cholesterol in mg/dl | Numerical
FBS | Fasting blood sugar > 120 mg/dl (1 = true; 0 = false) | Categorical
RestECG | Resting electrocardiogram results (0, 1, 2) | Categorical
Thalach | Maximum heart rate achieved | Numerical
Exang | Exercise-induced angina (1 = yes; 0 = no) | Categorical
Oldpeak | ST depression induced by exercise relative to rest | Numerical
Slope | Slope of the peak exercise ST segment | Numerical
CA | Number of major vessels (0-3) colored by fluoroscopy | Both numerical & categorical
Thal | Normal; fixed; reversible | Categorical
Target | Diagnosis of heart disease (1 = true; 0 = false) | Target

We will create a neural network with 2 layers (1 hidden layer and 1 output layer) to solve this problem.

**Q0**: This is a problem of ???

**Answer:** ___



## I. Importing Libraries

---



In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd # A useful library for reading CSV files

from tensorflow.keras import layers # Instead of writing tf.keras.layers.something, we can just write layers.something

## II. Preparing the Data
**QII.1)** Download the data and store it in a pandas DataFrame object.

In [ ]:
file_url = "http://storage.googleapis.com/download.tensorflow.org/data/heart.csv"
dataframe = pd.read_csv(file_url)
dataframe = dataframe.drop([247,252]) # Remove records 247 and 252, which have specific modalities

**QII.2)** Display the first 10 rows:

To avoid learning all the properties and methods of the DataFrame object, we will create numpy arrays.

We want to create an array `Y` containing the target variable (output, here `target`) and an array `X` containing the explanatory variables (inputs).

To extract a numpy array from the column `age`, for example, we write:

```python
age = np.array(dataframe.age)
```

**QII.3)** Let's start with the output. Create the variable `Y`:

**QII.4)** The variable `age` has high values, which may have more influence in the model than other variables like `sex`. It therefore needs to be normalized: subtract the mean `m` and divide by the standard deviation `e`. `norm_v = (v - m) / e`

The mean of a numpy array `a` is written as `np.mean(a)`. The standard deviation is written as `np.std(a)`.

Write the necessary lines to calculate the normalized `age`.

**QII.5)** Do the same for all other numerical variables:

`trestbps`, `chol`, `thalach`, `oldpeak`, `slope`, `ca`

To avoid repeating the same code six times, we will first create a `normalize` function that takes a DataFrame column as input and returns the normalized numpy array.

Then apply the function to all these variables.

**QII.6)** Display the first 5 values of `norm_ca`.

You should get:

```python
array([-0.72403274,  2.4808769 ,  1.41257369, -0.72403274, -0.72403274])
```

**QII.7)** The variables `sex`, `fbs`, and `exang` are binary variables (0 or 1) and thus do not need normalization.

Create the numpy arrays `sex`, `fbs`, and `exang`.

**QII.8)** The variables `cp`, `restecg`, and `thal` are categorical variables with more than 2 modalities. These modalities are not necessarily ordered, and it would be inefficient to input them directly into the model. We will use "one-hot encoding": we must create several binary variables, one for each possible modality. All these binary variables will be 0 except the one corresponding to the chosen modality, which will be 1. (See example here: https://en.wikipedia.org/wiki/One-hot)

An example code snippet for performing this operation is provided below:

In [ ]:
classList = ['aa', 'bb', 'cc'] # List of modalities
data = ['aa', 'aa', 'cc', 'bb', 'cc'] # Data in list format
OneHotEncoded = np.zeros((len(data), len(classList)), dtype=float) # Create a matrix of zeros with as many rows as patients and as many columns as possible modalities

for i in range(len(data)): # For each patient
  n = classList.index(data[i]) # Find the position of the modality data[i] in the classList (e.g., classList.index('cc') equals 2)
  OneHotEncoded[i, n] = 1. # Set the modality n for patient i to 1

print(OneHotEncoded)

[[1. 0. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]]


**Create a function to perform this operation and encode the 3 variables**: `cp`, `restecg`, and `thal`.

In [ ]:
def OneHotEncoding(data, classList):
  OneHotEncoded = np.zeros((len(data), len(classList)), dtype=float) # Create a matrix of zeros with as many rows as patients and as many columns as possible modalities

  for i in range(len(data)): # For each patient
    n = classList.index(data[i]) # Find the position of the modality data[i] in the classList (e.g., classList.index('cc') equals 2)
    OneHotEncoded[i, n] = 1. # Set the modality n for patient i to 1
  return OneHotEncoded

onehot_cp = OneHotEncoding(list(dataframe.cp), [0, 1, 2, 3, 4])
onehot_restecg = OneHotEncoding(list(dataframe.restecg), [0, 1, 2])
onehot_thal = OneHotEncoding(list(dataframe.thal), ['normal', 'fixed', 'reversible'])

**QII.9)** Display the first 5 rows of `onehot_thal`. You should get:

```
array([[0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       [1., 0., 0.]])
```


**QII.10)** We have preprocessed all the input variables. Now we need to concatenate them to create a large array `X`.

The command is `X = np.concatenate((X1, X2, ... Xn), axis=1)`
The variables `Xi` must be 2D arrays with 301 rows and 1 or more columns (1 column for numerical or binary variables, several columns for one-hot encoded variables).

When you type `norm_age.shape`, you will notice it is a 1D array.

What do you need to write to add a second dimension (refer to TP1 if you forgot)?

Create the large array `X` with the preprocessed data.

**QII.11)** Verify that `X` has dimensions 301 x 21.

**QII.12)** Now, we will split the data into three sets: training (60% = 181 samples), validation (20% = 60 samples), and testing (20% = 60 samples).

To do this, we create a random permutation of the patient indices. The first 181 will correspond to the training data, the next 60 to the validation data, and the last 60 to the testing data.

Complete the code below:

In [ ]:
index = np.arange(301) # 0, 1, 2, ..., 300
rng = np.random.default_rng(42) # 42 is a seed so everyone gets the same randomness. Why 42? Because it is the answer to the Ultimate Question of Life, the Universe, and Everything from *The Hitchhiker's Guide to the Galaxy*.
rng.shuffle(index) # `index` is now randomly shuffled, e.g., 209, 238, 61, 254, 166, ...

X_train, Y_train = X[index[???]], Y[index[???]]
X_val, Y_val = X[index[???]], Y[index[???]] # Hint: Do not start indices at 0, or validation data will overlap with training data
X_test, Y_test = X[index[???]], Y[index[???]]

**QII.13)** Verify that `X_test` has 60 rows and that the first two are:

```
[[ 0.15826117  1.          0.          0.          0.          0.
   1.          0.01785153 -1.20570916  0.          0.          0.
   1.         -1.93363519  1.          0.89117839  0.65701639  0.34427047
   0.          1.          0.        ]
 [-0.28487011  0.          0.          0.          0.          1.
   0.          0.24409622 -0.97538223  0.          0.          0.
   1.          0.85931287  0.         -0.82503847  0.65701639 -0.72403274
   1.          0.          0.        ]]
```


## III. Model Creation

**QIII.1)** Now we will create our model.
First, we create a fully connected layer with 32 neurons and ReLU activation.

Given the type of problem, what should be the number of neurons in the output layer and its activation function?
What will the model output correspond to?

Replace the two `???` placeholders in the code below to reflect your answer.

**Answer:** ____

In [ ]:
def create_model(): # For clarity, models are often defined in a function, but this is not mandatory
    I = layers.Input(batch_shape=(None, 21)) # 21 corresponds to the number of columns in `X`, None is for the batch size, left flexible
    L1 = layers.Dense(32, activation="relu")(I) # Each layer takes the previous layer as input
    output = layers.Dense(???, activation="???")(L1)
    return tf.keras.Model(I, output)

model = create_model()

**QIII.2)** To visualize the architecture, execute the following commands:

Why are there 704 weights for the first layer?

In [ ]:
model.summary()

**Answer:** __

**QIII.3)** The model must then be compiled. This is where we define the cost function and other metrics used to measure performance.

Since this is a classification problem, the cost function should be cross-entropy. As this is a binary classification problem with only one output (instead of two for general multiclass classification), we specifically use binary cross-entropy.

We also use accuracy (percentage of correct predictions, assuming a 50% probability threshold) as a more interpretable metric.

Additionally, we define the optimizer and learning rate.

In [ ]:
model.compile(
    tf.optimizers.Adam(learning_rate=1e-3),
    "binary_crossentropy",
    metrics=["accuracy"]
)

Finally, we train the model. Here, we use a batch size of 32, which is the default value.

In [ ]:
history = model.fit(
    X_train, Y_train,
    batch_size=32,
    epochs=100,
    validation_data=(X_val, Y_val)
)

**QIII.4)** What will be the number of batches for the training data? Where can you see this number when you execute the `model.fit(...)` command?

**Answer:**

You notice that the accuracy on the validation data increased and then decreased around 40 iterations. The accuracy on the training data, however, only increased.

The loss, which should be minimized, decreased until the end for the training data but not for the validation data.

**QIII.5)** Therefore, there was _____

We will plot the curves for better visualization:

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['accuracy']) # The variable `history` is the output of model.fit
plt.plot(history.history['val_accuracy'])
plt.legend(['train', 'validation'], loc='upper left')
plt.title('model accuracy')
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['train', 'validation'], loc='upper left')
plt.title('model loss')
plt.show()

III.6) We need to save the weights at the moment when the validation loss is at its minimum. Therefore, we will save whenever a new record is achieved.

In [ ]:
callbacks = [tf.keras.callbacks.ModelCheckpoint(filepath="model1/best-{epoch:05d}-{val_loss:.4f}.keras", save_best_only=True, verbose=1)]
model = create_model()
model.compile(tf.optimizers.Adam(learning_rate=1e-3),
              "binary_crossentropy",
              metrics=["accuracy"])
history = model.fit(X_train, Y_train,
          batch_size=32,
          epochs=100,
          validation_data=(X_val, Y_val),
          callbacks=callbacks)

**QIII.7)** Load the best model. In my case, it is `best-00054-0.3865.keras`, but you need to replace it with your own (check in the files panel on the left side of your screen).

In [ ]:
model = tf.keras.models.load_model('/content/model1/best-00054-0.3865.keras')


We will re-evaluate on the validation data. Verify that you obtain the same results as during training.

In [ ]:
loss, accuracy = model.evaluate(X_val, Y_val)
print("loss:", loss, " accuracy:", accuracy)

**QIII.8)** Perform the same evaluation on the training data. Are the performances better than on the validation data? What is this phenomenon called?

*Due to randomness, this phenomenon might not occur for you. Answer these questions as if the training loss is smaller than the validation loss, even if that's not the case for you.*

In [ ]:
# Command to evaluate on the training data:






**Answer:**

We want to see the model's output for each individual in the validation data and compare it with the target variable. The function to apply the model to data is `model.predict`, as shown below.

In [ ]:
Y_val_est = model.predict(X_val)
print(Y_val_est)


**QIII.9)** What exactly do these displayed values represent?

**Answer:**

**QIII.10)** Create a table to display `Y_val` in the first column and `Y_val_est` in the second column.

*Check the shapes of these two arrays beforehand.*

**QIII.11)** Now we will try a deeper but narrower architecture. Create:

- A dense layer with 12 neurons and ReLU activation
- A dense layer with 6 neurons and ReLU activation
- A dense layer with 1 neuron and Sigmoid activation

What loss do you obtain? There is always some randomness, so you can train it multiple times to see if, on average, you perform better.

*To avoid overwriting your previous model, name this one `model2` and create the function `create_model2`. Also replace it in the callback.*

**QIII.12)** Complete: The best validation loss obtained is ??? at iteration ???

**QIII.14)** Try to beat your validation data record by varying the activations, the number, or the size of the layers.
*Do not overwrite your previous models. Name them `model3`, `model4`, `model5`, etc., and create functions `create_model3`, `create_model4`, `create_model5`, etc.*

**QIII.15)** Once you are satisfied with your results, perform the final evaluation on the test data to get an unbiased estimate of your model's performance. You must not cheat; evaluate on the test data only after your model is fully validated on the validation data. If you make multiple attempts on the test data and take the best one, your performance estimate will be over-optimistic.

In [ ]:
model.load_weights('/content/M2-best-00047-0.3102.h5')
model.evaluate(X_test, Y_test)

Test data response: The best model has a cross-entropy of 0.44 and an accuracy of 80%.